In [ ]:
import fitz  # PyMuPDF
import os

# I read all the event PDFs and pull the raw text out of each one.
# I loop over the folder so I do not have to list the files by hand.
def extract_text_from_pdfs(folder_path):
    all_text = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):  # only take PDF files, skip anything else
            with fitz.open(os.path.join(folder_path, filename)) as doc:
                pdf_text = ""
                for page in doc:
                    pdf_text += page.get_text()  # read the text of one page
                all_text.append(pdf_text)  # keep one big string per PDF
    return all_text

# I keep the PDFs in a "documents" folder next to this notebook so the path
# works on any machine. Before I used my own Desktop path and it broke for Nika.
pdf_folder_path = "documents"
pdf_texts = extract_text_from_pdfs(pdf_folder_path)


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# The PDF text is too long to feed to the model at once, so I cut it into
# smaller chunks. I tried chunk_size=100 first but the chunks were too small
# and the answers had no context. 800 with a bit of overlap worked much better.
# The overlap has to be smaller than the chunk size or it errors out.
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)

all_chunks = []
for text in pdf_texts:
    chunks = splitter.split_text(text)  # split one PDF's text into chunks
    all_chunks.extend(chunks)  # put all chunks from all PDFs in one list


In [ ]:
# Quick check to see what the chunks look like before I build the vector store.
print(f"Documents passed to the model: {all_chunks}")


In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

# I turn each chunk into a vector with the same llama3.2 model I use for chat.
embedding_model = OllamaEmbeddings(model="llama3.2")

# FAISS is the search index. It lets me find the chunks that are closest to a
# question instead of sending all the text to the model every time.
vectorstore = FAISS.from_texts(all_chunks, embedding=embedding_model)

# I save it to disk so I do not have to re-embed all the PDFs on every run.
vectorstore.save_local("marbet_vectorstore")


In [ ]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate

# This is the model that writes the actual answer. base_url points to Ollama.
# I run it on localhost here. On the BUas server the URL was different.
llm = ChatOllama(
    base_url="http://localhost:11434", 
    model="llama3.2"
)

# The retriever is what pulls the matching chunks out of FAISS for a question.
retriever = vectorstore.as_retriever()

# This is the prompt I send to the model. I tell it to answer only from the
# context I give it, and I list the document types so it knows what it has.
# {context} gets the retrieved chunks, {question} gets the user question.
template = """
You are a helpful assistant. Answer the following question based on the context provided. 

Here is some context about the documents:
1. Activities and excursions
2. Packing list
3. Scenic Eclipse A-Z Guide
4. Tutorials and additional documents
5. Travel Itinerary

{context}

Now, answer the user's question:
{question}
"""

# This was my first, shorter prompt. I kept it because the longer one above
# with the document list gave better answers, but I wanted to remember it.
# template =  """
# You are a helpful assistant. Answer the question based on the following documents:

# {context}

# Now, answer the user's question:
# {question}
# """


prompt = ChatPromptTemplate.from_template(template)

# This chain ties it all together: it retrieves chunks, fills the prompt, and
# asks the model. I keep the source documents so I can check where an answer
# came from.
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)


In [ ]:
# Quick test of the retriever on its own (no model yet). I wanted to see if it
# returns the right chunks for a real question from the visa document.
docs = retriever.get_relevant_documents('What document must you upload during an ESTA or eTA application?')
for doc in docs:
    print(doc.page_content)

In [ ]:
# One-shot version: ask once, show the chunks it found, then show the answer.
# I used this to check the chunks and the answer side by side while testing.
query = input("Ask your question: ").strip()

if query:
    # show which chunks the retriever picked for this question
    docs = retriever.get_relevant_documents(query)
    for i, doc in enumerate(docs):
        print(f"Document {i+1}:", doc.page_content)

    # now run the full chain to get the written answer
    result = qa.invoke({"query": query})
    print("Answer:", result["result"])
else:
    print("No question was asked. Please try again.")

In [ ]:
# This is the real chatbot loop. It keeps asking until I type 'exit'.
while True:
    question = input("I'm here to help you! What is your question? (type 'exit' to quit): ")

    # stop the loop when the user wants to quit
    if question.lower() == 'exit':
        break

    # ask the chain and print just the answer text
    result = qa.invoke({"query": question})
    print(result["result"])

# I had a lot of print() debugging here while I was getting the chain to work
# (printing the retrieved docs and the raw result). I removed it once the
# {"query": question} call returned proper answers and I did not need it anymore.
